# Assignment 1 — NumPy, Pandas, Matplotlib & Seaborn

## Industry problem statements
**Problem 1 — Customer portfolio monitoring:** A retail bank receives large customer transaction and profile tables every day. Analysts must quickly summarize, clean, group and visualize the data before making service or sales decisions.

**Problem 2 — Campaign performance:** A bank wants to understand which customer segments respond to campaigns. Efficient array operations, DataFrame manipulation and visual analysis are required before any predictive model is built.

## Intuitive understanding
Think of **NumPy as the calculator**, **Pandas as the filing cabinet**, **Matplotlib as the chart paper**, and **Seaborn as the chart designer**. The assignment builds the foundation for everything that follows in data science.

## What are we trying to answer?
1. How do numerical arrays behave?
2. How do we inspect, clean, filter and aggregate tabular data?
3. How do visualisations turn rows of data into business patterns?
4. How can we make repeated calculations efficient?

## Dataset
We use a Titanic tabular dataset for the hands-on DataFrame work. The assignment explicitly allows public datasets including Titanic. The notebook also demonstrates general NumPy operations independently of the dataset.

## Success means
The notebook runs from top to bottom, demonstrates the requested operations, produces interpretable tables/plots, and explains each result in business language.

## Step 1 — Import important libraries

**Step briefing — What / Why / Expected output**

- **What we are doing:** Load NumPy, Pandas, Matplotlib and Seaborn before doing any analysis.
- **Why it matters:** These are the core tools requested by the assignment and prevent repeated imports later.
- **What the output should tell us:** A clean environment with all expected packages available.

**Retail-banking lens:** A banker should establish the analysis toolkit before touching customer data.


In [ ]:
# Core data handling
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("Libraries loaded successfully.")


## Step 2 — Load the dataset

**Step briefing — What / Why / Expected output**

- **What we are doing:** Load Titanic using Seaborn so the notebook is runnable in Colab without manual file paths.
- **Why it matters:** A reproducible dataset removes friction while demonstrating the same DataFrame workflow used on bank data.
- **What the output should tell us:** A populated DataFrame named `df`.

**Retail-banking lens:** In a bank, this is equivalent to loading a customer, transaction or campaign table.


In [ ]:
try:
    df = sns.load_dataset("titanic").copy()
    DATA_SOURCE = "Seaborn Titanic (assignment-compatible public dataset)"
except Exception as exc:
    # Local smoke-test fallback only. Replace with the official Titanic CSV for submission if required.
    rng = np.random.default_rng(RANDOM_STATE)
    n = 300
    df = pd.DataFrame({
        "survived": rng.integers(0,2,n),
        "pclass": rng.integers(1,4,n),
        "sex": rng.choice(["male","female"],n),
        "age": np.clip(rng.normal(32,14,n),1,80),
        "sibsp": rng.integers(0,4,n),
        "parch": rng.integers(0,4,n),
        "fare": np.clip(rng.lognormal(3.2,0.7,n),5,300),
        "embarked": rng.choice(["C","Q","S"],n),
        "class": rng.choice(["First","Second","Third"],n),
        "who": rng.choice(["man","woman","child"],n),
        "adult_male": rng.choice([True,False],n),
        "deck": rng.choice(["A","B","C","D","E",None],n),
        "embark_town": rng.choice(["Cherbourg","Queenstown","Southampton"],n),
        "alive": rng.choice(["yes","no"],n),
        "alone": rng.choice([True,False],n),
    })
    DATA_SOURCE = "Local smoke-test fallback (not the official submission dataset)"
print("Data source:", DATA_SOURCE)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

## Step 3 — Summarise the dataset

**Step briefing — What / Why / Expected output**

- **What we are doing:** Inspect rows, columns, data types, missing values, and descriptive statistics.
- **Why it matters:** Before transforming data, we need to know what we actually received.
- **What the output should tell us:** The dataset structure, quality issues and scale become visible.

**Retail-banking lens:** A banker would never approve a report without first checking row counts, field types and missing values.


In [ ]:
display(df.head())
print("\nShape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("\nDescriptive statistics:")
display(df.describe(include="all").T)

## Step 4 — Preprocess the dataset

**Step briefing — What / Why / Expected output**

- **What we are doing:** Handle duplicates and demonstrate missing-value treatment without overwriting the original analytical copy.
- **Why it matters:** Cleaning improves reliability and makes later operations predictable.
- **What the output should tell us:** A cleaner DataFrame and a visible audit of what was changed.

**Retail-banking lens:** This mirrors correcting incomplete customer records before reporting.


In [ ]:
df_clean = df.drop_duplicates().copy()

# Demonstration strategy:
# numeric -> median, categorical -> mode
for col in df_clean.select_dtypes(include=np.number).columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in df_clean.select_dtypes(exclude=np.number).columns:
    mode = df_clean[col].mode(dropna=True)
    if not mode.empty:
        df_clean[col] = df_clean[col].fillna(mode.iloc[0])

print("Duplicates after cleaning:", df_clean.duplicated().sum())
print("Total remaining missing values:", int(df_clean.isna().sum().sum()))


## Step 5 — NumPy array creation and operations

**Step briefing — What / Why / Expected output**

- **What we are doing:** Create 1D, 2D and 3D arrays; demonstrate arithmetic, slicing, broadcasting, reshaping and vectorisation.
- **Why it matters:** NumPy is designed for efficient numerical operations over arrays.
- **What the output should tell us:** The output shows how multiple values can be transformed in one operation.

**Retail-banking lens:** This is similar to calculating balances, ratios or monthly performance across thousands of customers at once.


In [ ]:
a1 = np.array([10, 20, 30, 40, 50])
a2 = np.arange(1, 13).reshape(3, 4)
a3 = np.arange(24).reshape(2, 3, 4)

print("1D array:", a1)
print("\n2D array:\n", a2)
print("\n3D shape:", a3.shape)

print("\nElement-wise addition:", a1 + 5)
print("Element-wise multiplication:", a1 * 2)
print("Slicing a1[1:4]:", a1[1:4])

broadcast_result = a2 + np.array([10, 20, 30, 40])
print("\nBroadcasting result:\n", broadcast_result)

print("\nVectorised square roots:", np.sqrt(a1))
print("\nSum by columns:", a2.sum(axis=0))


## Step 6 — Pandas selection, filtering, grouping and aggregation

**Step briefing — What / Why / Expected output**

- **What we are doing:** Select columns, filter rows, group records, aggregate numeric fields and sort results.
- **Why it matters:** This is how analysts turn raw rows into management information.
- **What the output should tell us:** Tables showing meaningful groups and summary measures.

**Retail-banking lens:** For a banker, grouping by class, sex or embarkation is analogous to grouping customers by segment, product or branch.


In [ ]:
selected = df_clean[["sex", "pclass", "age", "fare"]]
print("Selected columns:")
display(selected.head())

filtered = df_clean[(df_clean["age"] >= 30) & (df_clean["fare"] > df_clean["fare"].median())]
print("\nFiltered customers/passengers:")
display(filtered.head())

grouped = (
    df_clean.groupby("pclass")
    .agg(passenger_count=("pclass","size"),
         average_age=("age","mean"),
         average_fare=("fare","mean"))
    .sort_values("average_fare", ascending=False)
)
print("\nGrouped summary:")
display(grouped)


## Step 7 — Feature engineering and transformations

**Step briefing — What / Why / Expected output**

- **What we are doing:** Create derived fields and demonstrate normalization/log-style transformations.
- **Why it matters:** New variables can make patterns easier to analyse and can prepare data for modelling.
- **What the output should tell us:** New columns such as family size and normalized values.

**Retail-banking lens:** A banker commonly creates age bands, income-to-expense ratios or customer tenure bands from raw fields.


In [ ]:
df_fe = df_clean.copy()

df_fe["family_size"] = df_fe["sibsp"] + df_fe["parch"] + 1
df_fe["fare_log"] = np.log1p(df_fe["fare"])
df_fe["age_normalized"] = (
    (df_fe["age"] - df_fe["age"].min()) /
    (df_fe["age"].max() - df_fe["age"].min())
)

display(df_fe[["age","fare","family_size","fare_log","age_normalized"]].head())


## Step 8 — Visual analysis with Matplotlib and Seaborn

**Step briefing — What / Why / Expected output**

- **What we are doing:** Create the requested line, bar, scatter, histogram, heatmap and violin-style views where appropriate.
- **Why it matters:** Visuals expose distributions, group differences and relationships faster than raw tables.
- **What the output should tell us:** Each plot should answer a business question rather than exist only for decoration.

**Retail-banking lens:** A banker can use these visuals to compare customer segments, product uptake or campaign response.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_fe["age"].plot(kind="hist", bins=20, ax=axes[0,0], title="Age Distribution")
axes[0,0].set_xlabel("Age")

sns.barplot(data=df_fe, x="pclass", y="fare", estimator="mean", errorbar=None, ax=axes[0,1])
axes[0,1].set_title("Average Fare by Class")

sns.scatterplot(data=df_fe, x="age", y="fare", hue="survived", alpha=0.7, ax=axes[1,0])
axes[1,0].set_title("Age vs Fare")

sns.violinplot(data=df_fe, x="pclass", y="age", inner="quartile", ax=axes[1,1])
axes[1,1].set_title("Age Distribution by Class")

plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
numeric_cols = df_fe.select_dtypes(include=np.number).columns
sns.heatmap(df_fe[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


## Step 9 — Fine-tuning and optimisation

**Step briefing — What / Why / Expected output**

- **What we are doing:** Compare vectorised NumPy operations with a Python loop and demonstrate a compact memory audit.
- **Why it matters:** Vectorisation is generally faster and scales better than repeatedly looping through rows.
- **What the output should tell us:** A timing comparison and visibility into DataFrame memory usage.

**Retail-banking lens:** At bank scale, efficient operations can materially reduce reporting time and infrastructure cost.


In [ ]:
import time

values = np.arange(1_000_000)

start = time.perf_counter()
loop_result = [x * 2 for x in values]
loop_time = time.perf_counter() - start

start = time.perf_counter()
vector_result = values * 2
vector_time = time.perf_counter() - start

print(f"Loop time:       {loop_time:.6f} seconds")
print(f"Vectorised time: {vector_time:.6f} seconds")
print(f"Vectorised result correct: {np.array_equal(np.array(loop_result), vector_result)}")

print("\nMemory usage of cleaned DataFrame:")
display(df_clean.memory_usage(deep=True).sort_values(ascending=False).to_frame("bytes"))


## Final Project Review

**What problem did I solve?**  
I built the foundational workflow for numerical computation, tabular manipulation, preprocessing and visual analysis.

**What did the analysis/model learn?**  
This is a tooling assignment rather than a predictive model. The key learning is how structured data can be transformed into analysis-ready information.

**Which result matters most?**  
The most important result is the ability to move from raw records to grouped, cleaned and visualised information reproducibly.

**What limitation must be disclosed?**  
The Titanic dataset is a public learning dataset and is not representative of a modern retail-bank customer portfolio.

**What would I improve next?**  
Apply the same workflow to an actual banking portfolio dataset and add data validation, lineage and automated reporting.
